In [ ]:
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.sparse import csr_matrix

# multi- analysis ready

In [ ]:
def make_adata(df):
    obs = pd.DataFrame(index=df.index)
    obs['source'] = df['source']
    df_expr = df.drop(columns='source')
    adata = sc.AnnData(X=csr_matrix(df_expr.values), obs=obs, var=pd.DataFrame(index=df_expr.columns))
    adata.obs['n_counts'] = adata.X.sum(axis=1).A1
    adata.obs['n_genes'] = (adata.X > 0).sum(axis=1).A1
    return adata

In [ ]:
df_spatial = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/df_spatial.csv", index_col=0)

In [ ]:
adata_spatial = make_adata(df_spatial)
adata_spatial

In [ ]:
adata_spatial.obs['source']

In [ ]:
df_scrna = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/df_scrna.csv", index_col=0)

In [ ]:
import scanpy as sc
import pandas as pd
from scipy import io

In [ ]:
matrix = io.mmread('/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cord_nuc_data.mtx').T.tocsr()
cells = pd.read_csv('/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cord_nuc_cells.txt', header=None)[0].values
genes = pd.read_csv('/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/Cord_nuc_genes.txt', header=None)[0].values

adata_snrna = sc.AnnData(X=matrix)
adata_snrna.obs_names = cells
adata_snrna.var_names = genes

In [ ]:
adata_snrna.obs['n_counts'] = adata_snrna.X.sum(axis=1).A1
adata_snrna.obs['n_genes'] = (adata_snrna.X > 0).sum(axis=1).A1

In [ ]:
adata_snrna.obs['source'] = "snRNA"

In [ ]:
adata_snrna.obs['source']

In [ ]:
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_snrna.h5ad")

In [ ]:
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_spatial.h5ad")

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/MASLD_scrna01.h5ad")

In [ ]:
adata_scrna = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/MASLD_scrna01.h5ad")
adata_snrna = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_snrna.h5ad")
adata_spatial = sc.read("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_spatial.h5ad")

In [ ]:
# 1. float으로 변환
adata_spatial.X = adata_spatial.X.astype('float32')

# 2. normalize
import scanpy as sc
sc.pp.normalize_total(adata_spatial, target_sum=1e4)

# 3. log1p
sc.pp.log1p(adata_spatial)

In [ ]:
adata_scrna.obs['n_counts'] = adata_scrna.X.sum(axis=1).A1
adata_scrna.obs['n_genes'] = (adata_scrna.X > 0).sum(axis=1).A1
adata_snrna.obs['n_counts'] = adata_snrna.X.sum(axis=1).A1
adata_snrna.obs['n_genes'] = (adata_snrna.X > 0).sum(axis=1).A1
adata_spatial.obs['n_counts'] = adata_spatial.X.sum(axis=1).A1
adata_spatial.obs['n_genes'] = (adata_spatial.X > 0).sum(axis=1).A1

# scale

In [ ]:
sc.pp.scale(adata_scrna)
sc.tl.pca(adata_scrna)
sc.pp.neighbors(adata_scrna)
sc.tl.umap(adata_scrna)
sc.pl.umap(adata_scrna, color="source")

In [ ]:
sc.pp.scale(adata_spatial)
sc.tl.pca(adata_spatial)
sc.pp.neighbors(adata_spatial)
sc.tl.umap(adata_spatial)
sc.pl.umap(adata_spatial, color="source")

In [ ]:
sc.pp.scale(adata_snrna)
sc.tl.pca(adata_snrna)
sc.pp.neighbors(adata_snrna)
sc.tl.umap(adata_snrna)
sc.pl.umap(adata_snrna, color="source")

In [ ]:
sc.pp.scale(adata_spatial)
sc.tl.pca(adata_spatial)
sc.pp.neighbors(adata_spatial)
sc.tl.umap(adata_spatial)

In [ ]:
sc.pl.umap(adata_spatial, color="source")

In [ ]:
sc.pp.scale(adata_spatial)
sc.tl.pca(adata_spatial)
sc.pp.neighbors(adata_spatial)
sc.tl.umap(adata_spatial)
sc.pl.umap(adata_spatial, color="source")

In [ ]:
common_genes = set(adata_scrna.var_names) & set(adata_spatial.var_names)& set(adata_snrna.var_names)

In [ ]:
len(common_genes)

In [ ]:
adata_scrna = adata_scrna[:, list(common_genes)]
adata_spatial = adata_spatial[:, list(common_genes)]
adata_snrna = adata_snrna[:, list(common_genes)]

In [ ]:
adata_combined = adata_scrna.concatenate(adata_snrna, adata_spatial, batch_key='modality', batch_categories=['scRNA','snRNA', 'spatial'])

In [ ]:
adata_combined 

In [ ]:
sc.tl.pca(adata_combined)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)
sc.pl.umap(adata_combined, color="source")

In [ ]:
sc.tl.pca(adata_combined)
sc.pp.neighbors(adata_combined)
sc.tl.umap(adata_combined)
sc.pl.umap(adata_combined, color="source")

In [ ]:
import harmonypy as hm
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_combined.obsm['X_pca'],adata_combined.obs, 'source')
# Harmony 결과를 저장
adata_combined.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_combined, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_combined)
sc.pl.umap(adata_combined, color=['source'])

In [ ]:
import harmonypy as hm
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_combined.obsm['X_pca'],adata_combined.obs, 'source')
# Harmony 결과를 저장
adata_combined.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_combined, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_combined)
sc.pl.umap(adata_combined, color=['source'])

In [ ]:
sc.tl.leiden(adata_combined, resolution=0.7, key_added="leiden")
sc.pl.umap(adata_combined, color=['leiden'])

In [ ]:
rama_markers = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran.txt", sep="\t")
gene_list = list(rama_markers.iloc[:, 0])  # 첫 번째 column만 사용

In [ ]:
valid_genes = [gene for gene in gene_list if gene in adata_combined.var_names]

In [ ]:
sc.pl.dotplot(adata_combined, var_names=valid_genes, groupby="leiden", standard_scale='var', dot_max=0.5, show=True)

In [ ]:
rama_markers1 = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran_TableS1.txt", sep="\t")
gene_list1 = list(rama_markers1.iloc[:, 0])  # 첫 번째 column만 사용
valid_genes1 = [gene for gene in gene_list1 if gene in adata_combined.var_names]

In [ ]:
sc.pl.dotplot(adata_combined, var_names=valid_genes1, groupby="leiden", standard_scale='var', dot_max=0.5, show=True)

In [ ]:
pd.crosstab(adata_combined.obs['leiden'], adata_combined.obs['source'])

In [ ]:
adata_combined.obs['rama_annot'] = adata_combined.obs['leiden']

In [ ]:
adata_combined.obs['rama_annot'] = adata_combined.obs['rama_annot'].replace({
    '1': 'Epithelia',
    '6': 'Epithelia',
    '26':'Epithelia',
    '14': 'Mesenchyma',
    '15': 'Mesenchyma',
    '27': 'Mesenchyma',
    '10': 'Endothelia',
    '17': 'Endothelia',
    '25': 'Endothelia',
    '3': 'T&ILC',
    '4': 'T&ILC',
    '7': 'T&ILC',
    '8': 'T&ILC',
    '12': 'T&ILC',
    '21': 'T&ILC',
    '18': 'B',
    '23': 'B',
    '22': 'pDC',
    '5': 'MP',
    '13': 'MP',
    '28': 'MP',
    '18': 'Cycling'
    
})

sc.pl.umap(adata_combined, color=['rama_annot'])

In [ ]:
sc.pl.umap(adata_combined, color=['source'])

In [ ]:
print(adata_combined.obs['rama_annot'].value_counts())

In [ ]:
print(adata_combined.obs['rama_annot'].value_counts())

In [ ]:
adata_combined

# non-scale 이미 scale 되었으니 두번 scale 방지

In [ ]:
sc.tl.pca(adata_scrna)
sc.pp.neighbors(adata_scrna)
sc.tl.umap(adata_scrna)
sc.pl.umap(adata_scrna, color="source")

In [ ]:
sc.tl.pca(adata_snrna)
sc.pp.neighbors(adata_snrna)
sc.tl.umap(adata_snrna)
sc.pl.umap(adata_snrna, color="source")

In [ ]:
adata_spatial

In [ ]:
adata_spatial.X = adata_spatial.X.astype('float32')
sc.tl.pca(adata_spatial)
sc.pp.neighbors(adata_spatial)
sc.tl.umap(adata_spatial)
sc.pl.umap(adata_spatial, color="source")

In [ ]:
plt.plot(adata_scrna.uns['pca']['variance_ratio'][:50], marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
plt.plot(adata_snrna.uns['pca']['variance_ratio'][:50], marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
plt.plot(adata_spatial.uns['pca']['variance_ratio'][:50], marker='o')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
adata_scrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_scrna02.h5ad")
adata_snrna.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_snrna02.h5ad")
adata_spatial.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_spatial02.h5ad")

In [ ]:
common_genes = set(adata_scrna.var_names) & set(adata_spatial.var_names)& set(adata_snrna.var_names)
len(common_genes)

In [ ]:
adata_scsn = adata_scrna.concatenate(adata_snrna, batch_key='modality', batch_categories=['scRNA','snRNA'])
adata_scsp = adata_scrna.concatenate(adata_spatial, batch_key='modality', batch_categories=['scRNA','spatial'])
adata_snsp = adata_snrna.concatenate(adata_spatial, batch_key='modality', batch_categories=['snRNA','spatial'])

In [ ]:
adata_snsp

In [ ]:
import harmonypy as hm
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(adata_scsp.obsm['X_pca'],adata_scsp.obs, 'source')
# Harmony 결과를 저장
adata_scsp.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(adata_scsp, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(adata_scsp)
sc.pl.umap(adata_scsp, color=['source'])

In [ ]:
import bbknn
bbknn.bbknn(adata_scsp, batch_key='source')
sc.tl.umap(adata_scsp)

# Figure 생성
fig, ax = plt.subplots(figsize=(5, 5))

# Scanpy UMAP 플롯 (기존 색상 유지)
sc.pl.umap(
    adata_scsp, 
    color='source',
    size=20,
    alpha=0.8,
    legend_fontsize=12,
    #palette=loaded_color_dict,  # ✅ 기존 색상 적용
    ax=ax  
)

plt.show()

In [ ]:
sc.pl.umap(adata_scsp, color=['source'])

In [ ]:
sc.tl.leiden(adata_scsp, resolution=0.7, key_added="leiden")
sc.pl.umap(adata_scsp, color=['leiden'])

In [ ]:
sc.pl.umap(adata_scsp, color=['SLC7A5'])

In [ ]:
pd.crosstab(adata_scsp.obs['leiden'], adata_scsp.obs['source'])

In [ ]:
rama_markers = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran.txt", sep="\t")
gene_list = list(rama_markers.iloc[:, 0])  # 첫 번째 column만 사용
valid_genes = [gene for gene in gene_list if gene in adata_scsp.var_names]
sc.pl.dotplot(adata_scsp, var_names=valid_genes, groupby="leiden", standard_scale='var', dot_max=0.5, show=True)

In [ ]:
rama_markers1 = pd.read_csv("/home/jaehyunchoi/MASLD_collaboration/pre_work/Markers_heatmap_Ramachandran_TableS1.txt", sep="\t")
gene_list1 = list(rama_markers1.iloc[:, 0])  # 첫 번째 column만 사용
valid_genes1 = [gene for gene in gene_list1 if gene in adata_scsp.var_names]

In [ ]:
sc.pl.dotplot(adata_scsp, var_names=valid_genes1, groupby="leiden", standard_scale='var', dot_max=0.5, show=True)

In [ ]:
adata_scsp.write("/home/jaehyunchoi/MASLD_collaboration/multiomics_analysis/adata_scsp01.h5ad")

# NMF를 이용하여 투영후 진행

In [ ]:
from sklearn.decomposition import NMF
import torch
import numpy as np
from scipy.sparse import vstack

In [ ]:
adata_scrna

In [ ]:
adata_scrna = adata_scrna[:, list(common_genes)]
adata_spatial = adata_spatial[:, list(common_genes)]
adata_snrna = adata_snrna[:, list(common_genes)]

In [ ]:
# numpy array로 준비
X_scrna = torch.tensor(adata_scrna.X.toarray(), dtype=torch.float32)  # (cells_scRNA, genes)
X_spatial = torch.tensor(adata_spatial.X.toarray(), dtype=torch.float32)  # (cells_spatial, genes)

n_cells_scrna, n_genes = X_scrna.shape
n_cells_spatial, _ = X_spatial.shape

n_components = 20  # latent dimension

In [ ]:
X_scrna

# Poisson loss

In [ ]:
# Reconstruction
recon_scrna = torch.matmul(W_scrna, H)
recon_spatial = torch.matmul(W_spatial, H)

# Poisson negative log-likelihood
def poisson_loss(X, recon):
    return (recon - X * torch.log(recon + 1e-8)).sum()

loss_scrna = poisson_loss(X_scrna, recon_scrna)
loss_spatial = poisson_loss(X_spatial, recon_spatial)

total_loss = loss_scrna + loss_spatial

In [ ]:
optimizer = torch.optim.Adam([W, H], lr=1e-3)  # K까지 학습하려면 [W, H, K_scrna, K_spatial]

for epoch in range(1000):
    optimizer.zero_grad()
    loss_scrna = poisson_loss(X_scrna, torch.matmul(W[:n_cells_scrna, :] @ K_scrna, H))
    loss_spatial = poisson_loss(X_spatial, torch.matmul(W[n_cells_scrna:, :] @ K_spatial, H))
    total_loss = loss_scrna + loss_spatial
    total_loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.item():.4f}")

In [ ]:
# W: 공통 latent factor matrix (all cells, components)
W = torch.nn.Parameter(torch.rand(n_cells_scrna + n_cells_spatial, n_components))

# H: latent-to-gene weight (components, genes)
H = torch.nn.Parameter(torch.rand(n_components, n_genes))

# K_scrna, K_spatial: 플랫폼-specific weight (identity 또는 학습 가능)
K_scrna = torch.eye(n_components)  # 고정 (또는 학습 파라미터로 선언)
K_spatial = torch.eye(n_components)  # 고정 (또는 학습 파라미터로 선언)

In [ ]:
# W 분리
W_scrna = W[:n_cells_scrna, :] @ K_scrna
W_spatial = W[n_cells_scrna:, :] @ K_spatial

In [ ]:
W

# MSE loss

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

In [ ]:
# Data tensors
X_scrna = torch.tensor(adata_scrna.X.toarray(), dtype=torch.float32, device=device)
X_spatial = torch.tensor(adata_spatial.X.toarray(), dtype=torch.float32, device=device)

In [ ]:
# 기본 latent matrices
W = torch.nn.Parameter(torch.rand(n_cells_scrna + n_cells_spatial, n_components, device=device))
H = torch.nn.Parameter(torch.rand(n_components, n_genes, device=device))

# Optional platform-specific matrices (identity or learnable)
K_scrna = torch.eye(n_components, device=device)
K_spatial = torch.eye(n_components, device=device)

# Optimizer
optimizer = torch.optim.Adam([W, H], lr=1e-3)  # K까지 학습하려면 [W, H, K_scrna, K_spatial]

# Training loop
for epoch in range(1000):
    optimizer.zero_grad()
    
    # Split W
    W_scrna = W[:n_cells_scrna, :] @ K_scrna
    W_spatial = W[n_cells_scrna:, :] @ K_spatial
    
    # Reconstructions
    recon_scrna = torch.matmul(W_scrna, H)
    recon_spatial = torch.matmul(W_spatial, H)
    
    # ✅ MSE loss
    loss_scrna = torch.nn.functional.mse_loss(recon_scrna, X_scrna)
    loss_spatial = torch.nn.functional.mse_loss(recon_spatial, X_spatial)
    
    total_loss = loss_scrna + loss_spatial
    total_loss.backward()
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss.item():.4f}")

In [ ]:
W_scrna

In [ ]:
W_spatial

In [ ]:
W_combined = torch.cat([W_scrna, W_spatial], dim=0).detach().cpu().numpy()

In [ ]:
from sklearn.cluster import KMeans
n_clusters = 10
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(W_combined)
labels = kmeans.labels_

In [ ]:
import umap
umap_embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0).fit_transform(W_combined)

plt.scatter(umap_embedding[:,0], umap_embedding[:,1], c=labels, cmap='tab10', s=5)
plt.title("Combined scRNA + spatial latent clustering")
plt.show()

In [ ]:
obs_combined = pd.concat([adata_scrna.obs.copy(), adata_spatial.obs.copy()], axis=0)

In [ ]:
# AnnData 생성
adata_latent = anndata.AnnData(
    X=W_combined,                # cell × latent matrix
    obs=obs_combined.reset_index(drop=True)
)

# Optional: latent component 이름 붙이기
adata_latent.var_names = [f"latent_{i}" for i in range(W_combined.shape[1])]
adata_latent.obs_names = list(obs_combined.index)

print(adata_latent)

In [ ]:
sc.pp.neighbors(adata_latent, use_rep="X")  # X는 기본 latent matrix
sc.tl.umap(adata_latent)
sc.tl.leiden(adata_latent)

sc.pl.umap(adata_latent, color=['source'])

In [ ]:
sc.tl.leiden(adata_latent, resolution=1.0)
print(adata_latent.obs['leiden'])

In [ ]:
sc.pl.umap(adata_latent, color=['leiden'])

In [ ]:
adata_scsp

In [ ]:
adata_scsp.obs['latent_leiden'] = adata_latent.obs['leiden'].values

In [ ]:
adata_latent.obs['umap_leiden'] = adata_scsp.obs['leiden'].values

In [ ]:
sc.pl.umap(adata_latent, color=['umap_leiden'])

In [ ]:
sc.pl.umap(adata_scsp, color=['leiden'])

In [ ]:
sc.pl.umap(adata_scsp, color=['latent_leiden'])

# another trial